# CM3070 Final Project
# Model 2 (v2) — Fine-Tuning LEGAL-BERT for Clause Classification

This notebook implements the planned upgrade to Model 2: replacing the zero-shot DeBERTa NLI baseline (38.85% macro F1, below the 65% target) with a genuinely fine-tuned `nlpaueb/legal-bert-base-uncased` sequence classifier, trained on the CUAD employment clause subset. This is the single largest remaining technical objective identified in both the project's own evaluation and supervisor feedback on the preliminary report.

**Relationship to the zero-shot prototype:** the zero-shot notebook (`Prototype1_1_.ipynb`) remains the documented feasibility baseline and is not modified. This notebook produces a separate, fine-tuned classifier and reports a direct before/after comparison using the same metrics and the same 6 target clause types, so the improvement (or lack of one) is measurable rather than assumed.

# Cell 1 - Install Dependencies

In [ ]:
!pip install transformers torch datasets scikit-learn pandas -q

# Cell 2 - Import Libraries

In [ ]:
import re
import json
import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, classification_report

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: No GPU detected. Fine-tuning will be very slow on CPU.")
    print("Go to Runtime > Change runtime type > select a GPU.")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Cell 3 - Load CUAD and Extract Employment-Relevant Clauses

CUAD-QA frames each clause type as a question (e.g. *"Highlight the parts (if any) of this contract related to 'Non-Compete' that should be reviewed by a lawyer."*) against a contract `context`, with the answer being the relevant excerpt. This cell filters CUAD for the 6 question categories matching this project's target clause types, and extracts each non-empty answer span as one labelled training example.

**Expanded from the zero-shot prototype's 120 examples.** The preliminary report's prototype used 20 examples per type as a feasibility check; fine-tuning needs more. This cell pulls up to `MAX_PER_TYPE` examples per type (capped by whatever CUAD actually contains for that category) rather than a fixed small number.

In [ ]:
TARGET_CLAUSE_TYPES = {
    "compensation clause": ["compensation", "salary"],
    "termination clause": ["termination"],
    "confidentiality clause": ["confidentiality", "non-disclosure"],
    "non-compete clause": ["non-compete", "noncompete"],
    "intellectual property clause": ["intellectual property", "ip assignment", "assignment of ip"],
    "probation clause": ["probation", "trial period"],
}
LABEL_LIST = list(TARGET_CLAUSE_TYPES.keys())
LABEL_TO_ID = {label: i for i, label in enumerate(LABEL_LIST)}
MAX_PER_TYPE = 150


def match_clause_type(question_text):
    """Maps a CUAD question string to one of the 6 target types, or None."""
    q = question_text.lower()
    for label, keywords in TARGET_CLAUSE_TYPES.items():
        if any(kw in q for kw in keywords):
            return label
    return None


def extract_employment_clauses(cuad_dataset, max_per_type=MAX_PER_TYPE):
    """
    Walks CUAD-QA examples, matches each question to a target clause type,
    and extracts every non-empty answer span as a (text, label) example,
    capped at max_per_type per category.
    """
    counts = {label: 0 for label in LABEL_LIST}
    examples = []

    for row in cuad_dataset:
        label = match_clause_type(row["question"])
        if label is None or counts[label] >= max_per_type:
            continue
        answers = row.get("answers", {})
        texts = answers.get("text", []) if isinstance(answers, dict) else []
        for span in texts:
            span = span.strip()
            if len(span) < 20:  # skip near-empty/noise spans
                continue
            examples.append({"clause_text": span, "label": label})
            counts[label] += 1
            break  # one example per question row, to encourage source diversity

    return pd.DataFrame(examples), counts


try:
    cuad = load_dataset("theatticusproject/cuad-qa", split="train")
    clauses_df, type_counts = extract_employment_clauses(cuad)
    print("CUAD loaded from HuggingFace Hub.")
except Exception as e:
    print(f"Could not load CUAD from the Hub ({e}).")
    print("Falling back to a small local sample so the rest of the notebook still runs.")
    clauses_df = pd.DataFrame([
        {"clause_text": "Employee shall be paid a base salary of $85,000 per year, payable monthly.", "label": "compensation clause"},
        {"clause_text": "This Agreement may be terminated by either party upon 30 days written notice.", "label": "termination clause"},
        {"clause_text": "Employee agrees not to disclose any confidential information of the Company.", "label": "confidentiality clause"},
        {"clause_text": "Employee shall not engage in any competing business for 12 months post-termination.", "label": "non-compete clause"},
        {"clause_text": "All work product created during employment shall be the sole property of the Company.", "label": "intellectual property clause"},
        {"clause_text": "The first 90 days of employment shall constitute a probationary period.", "label": "probation clause"},
    ] * 10)
    type_counts = clauses_df["label"].value_counts().to_dict()

print(f"\nTotal examples: {len(clauses_df)}")
for label in LABEL_LIST:
    print(f"  {label}: {type_counts.get(label, 0)}")

# Cell 4 - Simple Data Augmentation (Synonym Replacement)

The preliminary report identified the small CUAD subset as a fine-tuning risk and planned data augmentation (back-translation, synonym replacement) to address it. This cell implements lightweight synonym replacement using WordNet: for each example, a small number of non-stopword tokens are randomly swapped for a synonym, generating one augmented variant per original example. This roughly doubles the training set size without needing an external translation API.

In [ ]:
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.corpus import wordnet


def synonym_augment(text, n_replacements=2):
    words = text.split()
    candidates = [i for i, w in enumerate(words) if w.isalpha() and len(w) > 3]
    random.shuffle(candidates)
    replaced = 0
    for idx in candidates:
        if replaced >= n_replacements:
            break
        synsets = wordnet.synsets(words[idx])
        lemmas = {l.name().replace('_', ' ') for s in synsets for l in s.lemmas()
                  if l.name().lower() != words[idx].lower()}
        if lemmas:
            words[idx] = random.choice(list(lemmas))
            replaced += 1
    return ' '.join(words)


augmented_rows = []
for _, row in clauses_df.iterrows():
    aug_text = synonym_augment(row["clause_text"])
    if aug_text != row["clause_text"]:
        augmented_rows.append({"clause_text": aug_text, "label": row["label"]})

augmented_df = pd.DataFrame(augmented_rows)
full_df = pd.concat([clauses_df, augmented_df], ignore_index=True).drop_duplicates(subset="clause_text")
print(f"Original examples:   {len(clauses_df)}")
print(f"Augmented examples:  {len(augmented_df)}")
print(f"Combined (deduped):  {len(full_df)}")

# Cell 5 - Train / Validation / Test Split

In [ ]:
full_df["label_id"] = full_df["label"].map(LABEL_TO_ID)

train_df, temp_df = train_test_split(full_df, test_size=0.3, random_state=42, stratify=full_df["label_id"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label_id"])

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print("\nTest set held out and untouched until final evaluation in Cell 8.")

# Cell 6 - Tokenise with the LEGAL-BERT Tokeniser

This is the same tokeniser (`nlpaueb/legal-bert-base-uncased`) demonstrated in the zero-shot prototype. Here it is actually used to prepare model input, not just to confirm legal-domain tokenisation as in the earlier notebook.

In [ ]:
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ClauseDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_length)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


train_dataset = ClauseDataset(train_df["clause_text"], train_df["label_id"], tokenizer)
val_dataset = ClauseDataset(val_df["clause_text"], val_df["label_id"], tokenizer)
test_dataset = ClauseDataset(test_df["clause_text"], test_df["label_id"], tokenizer)
print("Tokenised train/val/test datasets ready.")

# Cell 7 - Load LEGAL-BERT and Configure Fine-Tuning

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABEL_LIST)
).to(device)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"macro_precision": precision, "macro_recall": recall, "macro_f1": f1}


training_args = TrainingArguments(
    output_dir="./legalbert_clause_classifier",
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)
print("Trainer configured. Run Cell 8 to fine-tune.")

# Cell 8 - Fine-Tune and Evaluate on the Held-Out Test Set

Training runs against the validation set each epoch (for early-stopping-style model selection via `load_best_model_at_end`); the test set is only touched once, here, for the final reported metrics -- the same discipline as the zero-shot baseline's held-out CUAD evaluation, so the two numbers are comparable.

In [ ]:
trainer.train()

test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=-1)
test_labels = test_predictions.label_ids

print("=" * 60)
print("FINE-TUNED LEGAL-BERT -- TEST SET RESULTS")
print("=" * 60)
print(classification_report(test_labels, test_preds, target_names=LABEL_LIST, zero_division=0))

precision, recall, f1, support = precision_recall_fscore_support(
    test_labels, test_preds, average=None, labels=list(range(len(LABEL_LIST))), zero_division=0
)
results_df = pd.DataFrame({
    "Clause Type": LABEL_LIST,
    "Precision": [f"{p:.0%}" for p in precision],
    "Recall": [f"{r:.0%}" for r in recall],
    "F1 Score": [f"{f:.0%}" for f in f1],
    "Support": support,
})
macro_f1 = f1.mean()
print(results_df.to_string(index=False))
print(f"\nMacro-averaged F1: {macro_f1:.2%}")

# Cell 9 - Before / After Comparison Against the Zero-Shot Baseline

Direct comparison against the preliminary report's documented zero-shot result, using the same macro-F1 metric and the same 6 clause types, so any improvement is measured rather than assumed.

In [ ]:
ZERO_SHOT_BASELINE = {
    "compensation clause": 0.37, "termination clause": 0.62, "confidentiality clause": 0.36,
    "non-compete clause": 0.32, "intellectual property clause": 0.40, "probation clause": 0.26,
}
ZERO_SHOT_MACRO_F1 = 0.3885
TARGET_F1 = 0.65

comparison_df = pd.DataFrame({
    "Clause Type": LABEL_LIST,
    "Zero-Shot F1": [f"{ZERO_SHOT_BASELINE[l]:.0%}" for l in LABEL_LIST],
    "Fine-Tuned F1": [f"{v:.0%}" for v in f1],
    "Change": [f"{(v - ZERO_SHOT_BASELINE[l])*100:+.1f} pp" for l, v in zip(LABEL_LIST, f1)],
})

print("=" * 65)
print("ZERO-SHOT BASELINE vs FINE-TUNED LEGAL-BERT")
print("=" * 65)
print(comparison_df.to_string(index=False))
print(f"\nZero-shot macro F1:   {ZERO_SHOT_MACRO_F1:.2%}")
print(f"Fine-tuned macro F1:  {macro_f1:.2%}")
print(f"Target threshold:     {TARGET_F1:.0%}")
print(f"Change:               {(macro_f1 - ZERO_SHOT_MACRO_F1)*100:+.1f} percentage points")
status = "MEETS" if macro_f1 >= TARGET_F1 else "BELOW"
print(f"\nResult: fine-tuned model is {status} the {TARGET_F1:.0%} target.")
if macro_f1 < TARGET_F1:
    print("If still below target, consider: more CUAD examples per type (raise MAX_PER_TYPE),")
    print("more augmentation, more epochs, or a lower learning rate -- in that order of likely impact.")
print("=" * 65)

# Cell 10 - Save the Fine-Tuned Model

In [ ]:
trainer.save_model("./legalbert_clause_classifier_final")
tokenizer.save_pretrained("./legalbert_clause_classifier_final")
print("Saved to ./legalbert_clause_classifier_final")

try:
    import shutil
    shutil.make_archive("legalbert_clause_classifier_final", "zip", "./legalbert_clause_classifier_final")
    from google.colab import files
    files.download("legalbert_clause_classifier_final.zip")
except Exception as e:
    print(f"(Download skipped -- not running in Colab: {e})")

print("\nTo use this model in place of the zero-shot classifier in Model 2 / the")
print("Pipeline notebook, replace the zero-shot pipeline() call with:")
print('  from transformers import AutoModelForSequenceClassification, AutoTokenizer')
print('  model = AutoModelForSequenceClassification.from_pretrained("./legalbert_clause_classifier_final")')
print('  tokenizer = AutoTokenizer.from_pretrained("./legalbert_clause_classifier_final")')
print('  # then argmax(model(**tokenizer(text, return_tensors="pt")).logits) for a prediction')

# Cell 11 - Summary

In [ ]:
print("=" * 60)
print("MODEL 2 (v2) SUMMARY -- LEGAL-BERT FINE-TUNING")
print("=" * 60)
print(f"Training examples (incl. augmentation): {len(train_df)}")
print(f"Validation examples:                    {len(val_df)}")
print(f"Test examples (held out):                {len(test_df)}")
print(f"Zero-shot baseline macro F1:              {ZERO_SHOT_MACRO_F1:.2%}")
print(f"Fine-tuned macro F1:                       {macro_f1:.2%}")

print("\n--- What this notebook demonstrates ---")
print("  1. LEGAL-BERT actually performing classification (not just tokenising),")
print("     replacing the zero-shot DeBERTa placeholder used in the prototype")
print("  2. Expanded CUAD extraction beyond the prototype's 20-per-type sample")
print("  3. WordNet-based synonym augmentation, as planned in the preliminary report")
print("  4. A held-out test set never touched until final evaluation")
print("  5. Direct, same-metric comparison against the documented zero-shot baseline")

print("\n--- Known limitations ---")
print("  - CUAD examples are drawn from general commercial contracts (EDGAR filings),")
print("    not employment contracts specifically -- some domain mismatch is possible")
print("  - Synonym augmentation has no word-sense disambiguation, so it can pick")
print("    an unrelated WordNet sense (e.g. 'business' -> 'stage', in the theatrical")
print("    sense, or a number word replaced with its Roman numeral) -- confirmed")
print("    during testing, not just a theoretical risk. Spot-check augmented examples")
print("    before trusting them; back-translation (also planned) would likely avoid")
print("    this failure mode but requires an external translation API")
print("  - This notebook has not yet been run against a GPU runtime as part of this")
print("    project's own testing -- actual achieved F1 should be verified in Colab")
print("    before being reported as a final result")

print("\n--- Next steps ---")
print("  - Run this notebook in Colab with a GPU runtime and record the real result")
print("  - If the target F1 is met, swap this fine-tuned model into the Pipeline")
print("    notebook in place of the zero-shot classifier")
print("  - Re-run the Pipeline's real-contract test with the fine-tuned model and")
print("    compare confidence/accuracy against the zero-shot results already documented")
print("=" * 60)